In [1]:
import os
import sys
from pathlib import Path
import shutil
sys.path.insert(0, '/home/mwalker/git/neoexchange/neoexchange') #point to top of file path
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'neox.settings')
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

import django
django.setup()
from django.conf import settings

import pandas as pd
import numpy as np
from IPython.display import display

from core.models import Body #Body Class
from core.models.sources import StaticSource #static class
from core.models.blocks import Block  #Block class
from core.models.frame import Frame 


os.chdir("/home/mwalker/git/neoexchange/neoexchange") #temp fix for json issue
from core.views import summarize_block_quality
from core.views import GuideMovie
from core.plots import generalized_fwhm_plotter, generalized_zeropoint_plotter  #get pngs for fwhm and zp to analyze


In [2]:
# Query the didymos static object
didymos = Body.objects.get(name = '65803') #object.get is digano model.model call

print(didymos)

#Query fields
fields = StaticSource.objects.filter(source_type = StaticSource.REFERENCE_FIELD, name__contains = '2026') #reference field is an attribute from the source_type attribute, gets attached via django


Tues_table_progress = {
    "field_name": None,
    "ra": None,
    "dec": None,


}
Data_table_tues = pd.DataFrame([
    {"field_name": field.name, 
     "ra": field.ra,
     "dec": field.dec}
 for field in fields])



print(Data_table_tues)

65803 is active
                    field_name          ra        dec
0   Didymos COJ 2026 Field #01  241.405425 -22.294570
1   Didymos COJ 2026 Field #02  241.146390 -22.280000
2   Didymos COJ 2026 Field #03  240.899405 -22.266720
3   Didymos COJ 2026 Field #04  240.664645 -22.254830
4   Didymos COJ 2026 Field #05  240.442260 -22.244395
5   Didymos COJ 2026 Field #06  240.232400 -22.235505
6   Didymos COJ 2026 Field #07  240.035180 -22.228215
7   Didymos COJ 2026 Field #08  239.850720 -22.222605
8   Didymos COJ 2026 Field #09  239.678785 -22.218705
9   Didymos COJ 2026 Field #10  239.520055 -22.216610
10  Didymos COJ 2026 Field #11  239.374230 -22.216335
11  Didymos COJ 2026 Field #12  239.241075 -22.217930
12  Didymos COJ 2026 Field #13  239.121095 -22.221425
13  Didymos COJ 2026 Field #14  239.013985 -22.226840
14  Didymos COJ 2026 Field #15  238.919705 -22.234195
15  Didymos COJ 2026 Field #16  238.838190 -22.243515


In [3]:
#NEXT: DO The Blocks


b_act_status= []
for field in fields:
    block_activity =list(Block.objects.filter(calibsource = field))
    b_act_status.append( {"field_name": field.name,
                          "block Status": [str(b) for b in block_activity] } )


status_table = pd.DataFrame(b_act_status)
print(status_table)

#check activity for didymos #6
act_5 = b_act_status[5]["block Status"]
print(act_5)

                    field_name  \
0   Didymos COJ 2026 Field #01   
1   Didymos COJ 2026 Field #02   
2   Didymos COJ 2026 Field #03   
3   Didymos COJ 2026 Field #04   
4   Didymos COJ 2026 Field #05   
5   Didymos COJ 2026 Field #06   
6   Didymos COJ 2026 Field #07   
7   Didymos COJ 2026 Field #08   
8   Didymos COJ 2026 Field #09   
9   Didymos COJ 2026 Field #10   
10  Didymos COJ 2026 Field #11   
11  Didymos COJ 2026 Field #12   
12  Didymos COJ 2026 Field #13   
13  Didymos COJ 2026 Field #14   
14  Didymos COJ 2026 Field #15   
15  Didymos COJ 2026 Field #16   

                                         block Status  
0   [4244099 is not active, 4242007 is not active,...  
1   [4247146 is not active, 4244100 is not active,...  
2   [4247110 is not active, 4244101 is not active,...  
3   [4252701 is not active, 4247111 is not active,...  
4   [4253584 is not active, 4253172 is not active,...  
5   [4253588 is not active, 4253173 is not active,...  
6   [4253174 is not active, 4

In [4]:
#Get number of Blocks and obs.blocks for each field
num_observed = []
for f in fields:
    blocks = Block.objects.filter(calibsource = f)
    for b in blocks:
        num_obs = b.num_observed
    num_observed.append({"field_name": f ,
                 "num_obs": num_obs} )

print(num_observed)

[{'field_name': <StaticSource: Didymos COJ 2026 Field #01 (Reference field)>, 'num_obs': None}, {'field_name': <StaticSource: Didymos COJ 2026 Field #02 (Reference field)>, 'num_obs': None}, {'field_name': <StaticSource: Didymos COJ 2026 Field #03 (Reference field)>, 'num_obs': None}, {'field_name': <StaticSource: Didymos COJ 2026 Field #04 (Reference field)>, 'num_obs': None}, {'field_name': <StaticSource: Didymos COJ 2026 Field #05 (Reference field)>, 'num_obs': None}, {'field_name': <StaticSource: Didymos COJ 2026 Field #06 (Reference field)>, 'num_obs': None}, {'field_name': <StaticSource: Didymos COJ 2026 Field #07 (Reference field)>, 'num_obs': None}, {'field_name': <StaticSource: Didymos COJ 2026 Field #08 (Reference field)>, 'num_obs': None}, {'field_name': <StaticSource: Didymos COJ 2026 Field #09 (Reference field)>, 'num_obs': None}, {'field_name': <StaticSource: Didymos COJ 2026 Field #10 (Reference field)>, 'num_obs': None}, {'field_name': <StaticSource: Didymos COJ 2026 Fi

In [5]:
#count number of blocks within the list of dictionaries

dict_status = b_act_status[1]["block Status"]

print(len(dict_status))

status_table["blocks"] = [len(item["block Status"]) for item in b_act_status]

Data_table_tues["blocks"] = [len(item["block Status"]) for item in b_act_status]


print(Data_table_tues)

22
                    field_name          ra        dec  blocks
0   Didymos COJ 2026 Field #01  241.405425 -22.294570      22
1   Didymos COJ 2026 Field #02  241.146390 -22.280000      22
2   Didymos COJ 2026 Field #03  240.899405 -22.266720      22
3   Didymos COJ 2026 Field #04  240.664645 -22.254830      23
4   Didymos COJ 2026 Field #05  240.442260 -22.244395      22
5   Didymos COJ 2026 Field #06  240.232400 -22.235505      15
6   Didymos COJ 2026 Field #07  240.035180 -22.228215      21
7   Didymos COJ 2026 Field #08  239.850720 -22.222605      15
8   Didymos COJ 2026 Field #09  239.678785 -22.218705      14
9   Didymos COJ 2026 Field #10  239.520055 -22.216610      20
10  Didymos COJ 2026 Field #11  239.374230 -22.216335      20
11  Didymos COJ 2026 Field #12  239.241075 -22.217930      14
12  Didymos COJ 2026 Field #13  239.121095 -22.221425      13
13  Didymos COJ 2026 Field #14  239.013985 -22.226840      14
14  Didymos COJ 2026 Field #15  238.919705 -22.234195      13
15  D

In [6]:
#get obs.blocks columns and ref frames
#ask about how to get 0's and 1' for the obs.blocks // ref reframes
obs_blockcol = []
ref_frames = []
num_obs = False
for f in fields:
    blocks = Block.objects.filter(calibsource =f)
    for b in blocks:
        if b.num_observed:
            num_obs = True
            break               #found observation
    if num_obs:
        ref_frames.append({"field":f,
                           "ref frames": "" })
        obs_blockcol.append({"field": f ,
                    "obs.blocks": sum(1 for b in blocks if b.num_observed)})
    if not num_obs:
        ref_frames.append({"field":f,
                           "ref frames": "" })
        obs_blockcol.append({"field": f ,
                             "obs.blocks": 0})
    
obs_0_5_num = [item["obs.blocks"] for item in obs_blockcol[0:5] ]
print(obs_0_5_num)




[1, 2, 2, 0, 3]


In [7]:
from datetime import date
didymos = Body.objects.get(name='65803')

blocks_20260708 = Block.objects.filter(
    body=didymos,
    block_start__gte=date(2026,7,8),
    block_start__lt=date(2026,7,13)
)

for b in blocks_20260708:
    print(
        "id:", b.id,
        "request:", b.request_number,
        "date:", b.block_start,
        "body:", b.body.current_name()
    )

    frames = Frame.objects.filter(
        block=b,
        frametype=Frame.NEOX_SUB_FRAMETYPE
    )

    for f in frames:
        print(f.filename)

id: 35906 request: 4253588 date: 2026-07-12 00:00:00 body: 65803
coj2m002-ep09-20260712-0136-e93.fits
coj2m002-ep09-20260712-0135-e93.fits
coj2m002-ep09-20260712-0134-e93.fits
coj2m002-ep09-20260712-0133-e93.fits
coj2m002-ep09-20260712-0132-e93.fits
coj2m002-ep09-20260712-0131-e93.fits
coj2m002-ep09-20260712-0130-e93.fits
coj2m002-ep09-20260712-0129-e93.fits
coj2m002-ep09-20260712-0128-e93.fits
coj2m002-ep09-20260712-0127-e93.fits
coj2m002-ep09-20260712-0126-e93.fits
coj2m002-ep09-20260712-0125-e93.fits
coj2m002-ep09-20260712-0124-e93.fits
coj2m002-ep09-20260712-0123-e93.fits
coj2m002-ep09-20260712-0122-e93.fits
coj2m002-ep09-20260712-0121-e93.fits
coj2m002-ep09-20260712-0120-e93.fits
coj2m002-ep09-20260712-0119-e93.fits
coj2m002-ep09-20260712-0118-e93.fits
coj2m002-ep09-20260712-0117-e93.fits
coj2m002-ep09-20260712-0116-e93.fits
coj2m002-ep09-20260712-0115-e93.fits
coj2m002-ep09-20260712-0114-e93.fits
coj2m002-ep09-20260712-0113-e93.fits
coj2m002-ep09-20260712-0112-e93.fits
coj2m002-e

In [8]:
didymos = Body.objects.get(name='65803')

all_blocks = Block.objects.filter(body=didymos)

print("Total Didymos blocks:", all_blocks.count())

for b in all_blocks[:10]:
    print(
        b.id,
        b.request_number,
        b.block_start
    )

Total Didymos blocks: 553
35906 4253588 2026-07-12 00:00:00
32642 3787422 2025-02-25 05:20:00
32640 3787420 2025-02-23 05:20:00
32639 3787419 2025-02-22 05:20:00
32641 3787421 2025-02-24 05:20:00
32573 3786750 2025-02-21 05:20:00
32540 3785896 2025-02-20 05:20:00
32309 3765325 2025-01-31 05:10:00
32276 3764719 2025-01-30 05:10:00
32242 3763961 2025-01-29 05:10:00


In [9]:
#frames ----RUN THIS!!!!!

Frames = Frame.objects.filter(frametype = Frame.REFERENCE_FRAMETYPE, block__block_start="2026-06-24")


ref_frames_list = []

for f in fields:
    blocks = list(Block.objects.filter(calibsource = f))
    ref_num = 0
    for b in blocks:

        #count ref frames
        Frames = Frame.objects.filter(block = b, frametype = Frame.REFERENCE_FRAMETYPE).count()
        ref_num += Frames

    ref_frames_list.append({"field name": f.name ,
                    "blocks": len(blocks) ,
                    "# ref frames": ref_num})
    
Data_table_tues["# ref frames"] = [ item["# ref frames"] for item in ref_frames_list]
          
print(pd.DataFrame(Data_table_tues))

                    field_name          ra        dec  blocks  # ref frames
0   Didymos COJ 2026 Field #01  241.405425 -22.294570      22             0
1   Didymos COJ 2026 Field #02  241.146390 -22.280000      22             3
2   Didymos COJ 2026 Field #03  240.899405 -22.266720      22             3
3   Didymos COJ 2026 Field #04  240.664645 -22.254830      23             0
4   Didymos COJ 2026 Field #05  240.442260 -22.244395      22             7
5   Didymos COJ 2026 Field #06  240.232400 -22.235505      15             8
6   Didymos COJ 2026 Field #07  240.035180 -22.228215      21             4
7   Didymos COJ 2026 Field #08  239.850720 -22.222605      15             4
8   Didymos COJ 2026 Field #09  239.678785 -22.218705      14             4
9   Didymos COJ 2026 Field #10  239.520055 -22.216610      20             0
10  Didymos COJ 2026 Field #11  239.374230 -22.216335      20             8
11  Didymos COJ 2026 Field #12  239.241075 -22.217930      14             4
12  Didymos 

In [10]:
for f in fields:
    blocks = list(Block.objects.filter(calibsource=f))
    print("field", f.name, "blocks", len(blocks))
    for b in blocks[:5]:
        total = Frame.objects.filter(block=b).count()
        ref = Frame.objects.filter(block=b, frametype=Frame.REFERENCE_FRAMETYPE).count()
        print("  block", b.id, "total frames", total, "reference frames", ref)
        if total and not ref:
            print("   frametypes:", list(Frame.objects.filter(block=b).values_list("frametype", flat=True)[:10]))

field Didymos COJ 2026 Field #01 blocks 22
  block 35696 total frames 0 reference frames 0
  block 35656 total frames 0 reference frames 0
  block 35616 total frames 0 reference frames 0
  block 35576 total frames 0 reference frames 0
  block 35536 total frames 0 reference frames 0
field Didymos COJ 2026 Field #02 blocks 22
  block 35772 total frames 156 reference frames 3
  block 35697 total frames 0 reference frames 0
  block 35657 total frames 0 reference frames 0
  block 35617 total frames 0 reference frames 0
  block 35577 total frames 0 reference frames 0
field Didymos COJ 2026 Field #03 blocks 22
  block 35736 total frames 156 reference frames 3
  block 35698 total frames 0 reference frames 0
  block 35658 total frames 0 reference frames 0
  block 35618 total frames 0 reference frames 0
  block 35578 total frames 0 reference frames 0
field Didymos COJ 2026 Field #04 blocks 23
  block 35805 total frames 0 reference frames 0
  block 35737 total frames 0 reference frames 0
  block 

In [11]:
fieldst = list(StaticSource.objects.filter(source_type = StaticSource.REFERENCE_FIELD, name__contains = '2026')) #reference field is an attribute from the source_type attribute, gets attached via django

blocks6 = list(Block.objects.filter(calibsource=fieldst[5]))
ref_frame = Frame.objects.filter(block__in = blocks6 ,frametype = Frame.REFERENCE_FRAMETYPE)
print("  block", b.id, "total frames", total, "reference frames", ref)

  block 35127 total frames 0 reference frames 0


In [12]:
##check Block UID ###
obs_block_15 = Block.objects.filter(calibsource= fields[14], num_observed__gte=1) 
block = obs_block_15[0]

block_15_IUDS = []
for blocks in obs_block_15:
    block_15_IUDS.append(blocks.get_blockuid)

print(block_15_IUDS)

[['812982002'], ['812964819']]


In [13]:
###FINAL TABLE###
pd.set_option("display.max_columns", None)

Data_table_tues["obs.block"] = [item["obs.blocks"] for item in obs_blockcol]



Data_table_tues["ra"] = [round(i, 2) for i in Data_table_tues["ra"]]
Data_table_tues["dec"] = [round(i, 2) for i in Data_table_tues["dec"]]
Data_table_tues["# ref frames"] = [ item["# ref frames"] for item in ref_frames_list]

print(display(Data_table_tues))

,field_name,ra,dec,blocks,# ref frames,obs.block
0,Didymos COJ 2026 Field #01,241.41,-22.29,22,0,1
1,Didymos COJ 2026 Field #02,241.15,-22.28,22,3,2
2,Didymos COJ 2026 Field #03,240.90,-22.27,22,3,2
3,Didymos COJ 2026 Field #04,240.66,-22.25,23,0,0
4,Didymos COJ 2026 Field #05,240.44,-22.24,22,7,3
5,Didymos COJ 2026 Field #06,240.23,-22.24,15,8,4
6,Didymos COJ 2026 Field #07,240.04,-22.23,21,4,2
7,Didymos COJ 2026 Field #08,239.85,-22.22,15,4,2
8,Didymos COJ 2026 Field #09,239.68,-22.22,14,4,1
9,Didymos COJ 2026 Field #10,239.52,-22.22,20,0,0


None


In [14]:
#find reference fileds from this disk ---Tim's code
reference_library = Path('/apophis/eng/rocks/reference_library/')


all_ref_filenames = []
field_name_test = []

for field in fields:
     field_name = f'{field.ra:.2f}_{field.dec:+.2f}'
     ref_filepaths = reference_library.glob(f'*{field_name}*')
     ref_filenames = [f.name for f in ref_filepaths if 'rms' not in f.name]
     
     all_ref_filenames.append(ref_filenames)
     field_name_test.append(field_name)
     
     filters = []
     for ref_filename in ref_filenames:
        chunks = ref_filename.split('_')
        filters.append(chunks[3])
        ref_filters_str = ",".join(filters) #filters


# extract field 15 reference images
num_ref_files6 = []
ref_files_names6 = []

for field in fields:
    chunks_fields = field.name.split(" ")
    if "#15" in chunks_fields:
       field_name = f'{field.ra:.2f}_{field.dec:+.2f}'
       ref_filepaths = reference_library.glob(f'*{field_name}*') # selects filenames with that ra and dec from the OG query
       ref_filenames = [f.name for f in ref_filepaths if 'rms' not in f.name]
       ref_files_names6.append(ref_filenames)
       num_ref_files6.append(len(ref_filenames))



print("In {} the reference files are: {} \n --------------------\n" 
      "And the Number of files is {} ".format(fields[5].name, ref_files_names6, num_ref_files6))

In Didymos COJ 2026 Field #06 the reference files are: [['reference_coj_ep07_rp_238.92_-22.23_20260624.fits', 'reference_coj_ep06_gp_238.92_-22.23_20260624.fits', 'reference_coj_ep08_ip_238.92_-22.23_20260624.fits', 'reference_coj_ep09_zs_238.92_-22.23_20260624.fits']] 
 --------------------
And the Number of files is [4] 


In [15]:
reference_library = Path('/apophis/eng/rocks/reference_library/')
output_root = Path('/home/mwalker/git/Combined_ref_images')

for field in fields:

    # Extract the field number from the name
    field_num = field.name.split("#")[-1].strip()

    # Create output directory (f1, f2, ..., f16)
    outdir = output_root / f"f{field_num}"
    outdir.mkdir(parents=True, exist_ok=True)

    # Find matching reference files
    field_name = f'{field.ra:.2f}_{field.dec:+.2f}'
    ref_filepaths = [
        f for f in reference_library.glob(f'*{field_name}*')
        if 'rms' not in f.name
    ]

    print(f"{field.name}: {len(ref_filepaths)} reference images")

    # Copy each file into the directory
    for ref_file in ref_filepaths:
        shutil.copy2(ref_file, outdir / ref_file.name)

Didymos COJ 2026 Field #01: 0 reference images
Didymos COJ 2026 Field #02: 3 reference images
Didymos COJ 2026 Field #03: 3 reference images
Didymos COJ 2026 Field #04: 0 reference images
Didymos COJ 2026 Field #05: 7 reference images
Didymos COJ 2026 Field #06: 8 reference images
Didymos COJ 2026 Field #07: 4 reference images
Didymos COJ 2026 Field #08: 4 reference images
Didymos COJ 2026 Field #09: 4 reference images
Didymos COJ 2026 Field #10: 0 reference images
Didymos COJ 2026 Field #11: 8 reference images
Didymos COJ 2026 Field #12: 4 reference images
Didymos COJ 2026 Field #13: 4 reference images
Didymos COJ 2026 Field #14: 4 reference images
Didymos COJ 2026 Field #15: 4 reference images
Didymos COJ 2026 Field #16: 4 reference images


In [16]:
field_name_to_check = 'reference_coj_ep07_rp_238.92_-22.23_20260624.fits'

In [17]:
print(all_ref_filenames)
print(ref_files_names6)
print(field_name_test)

for f in fields:
    field_name = f"{f.ra:.2f}_{f.dec:+.2f}"
    for path in list(reference_library.glob(f'*{field_name}')): #list over generator
        print(path)


[[], ['reference_coj_ep06_gp_241.15_-22.28_20260708.fits', 'reference_coj_ep08_ip_241.15_-22.28_20260708.fits', 'reference_coj_ep09_zs_241.15_-22.28_20260708.fits'], ['reference_coj_ep07_rp_240.90_-22.27_20260708.fits', 'reference_coj_ep06_gp_240.90_-22.27_20260708.fits', 'reference_coj_ep08_ip_240.90_-22.27_20260708.fits'], [], ['reference_coj_ep06_gp_240.44_-22.24_20260708.fits', 'reference_coj_ep07_rp_240.44_-22.24_20260708.fits', 'reference_coj_ep08_ip_240.44_-22.24_20260708.fits', 'reference_coj_ep06_gp_240.44_-22.24_20260710.fits', 'reference_coj_ep08_ip_240.44_-22.24_20260710.fits', 'reference_coj_ep07_rp_240.44_-22.24_20260710.fits', 'reference_coj_ep09_zs_240.44_-22.24_20260710.fits'], ['reference_coj_ep06_gp_240.23_-22.24_20260710.fits', 'reference_coj_ep08_ip_240.23_-22.24_20260710.fits', 'reference_coj_ep07_rp_240.23_-22.24_20260710.fits', 'reference_coj_ep09_zs_240.23_-22.24_20260710.fits', 'reference_coj_ep06_gp_240.23_-22.24_20260624.fits', 'reference_coj_ep08_ip_240.23_

In [18]:
#next: make this process of table generation and file call into a script--put that on your github
#connect via ssh to your computer

### Get Block Status ###

In [19]:
from astropy.table import vstack

sum_instance = GuideMovie()

astro_quality_table = []

for f in fields:
    obs_blocks = Block.objects.filter(calibsource = f, num_observed__gte =1)
    
    for b in obs_blocks:

       
        dataroot = os.path.join(settings.DATA_ROOT, 'Hera', b.get_blockdayobs)
        t =summarize_block_quality(dataroot, b)
        
        if t is not None:
            t["Field Name"] = np.full(len(t), f.name)
            t["# block"] = np.full(len(t), b.id)
            astro_quality_table.append(t)

master_astro_qual_table = vstack(astro_quality_table)





#organize quality by rms, fwhm, errors 


35208: 4224769 812320693
35772: 4247146 818294035
3255625        coj2m002-ep06-20260708-0983-e92.fits: 2026-07-08T14:10:16 gp 92  24.1934 +/-   0.0124 FWHM=2.264 RMS=  0.14 (#fit stars= 361 xy_c=1.66 as_c=12.96 #refstars=862)
3255621        coj2m002-ep06-20260708-0984-e92.fits: 2026-07-08T14:13:51 gp 92  24.2406 +/-   0.0134 FWHM=2.216 RMS=  0.14 (#fit stars= 390 xy_c=1.98 as_c=13.29 #refstars=948)
3255620        coj2m002-ep06-20260708-0985-e92.fits: 2026-07-08T14:17:26 gp 92  24.2905 +/-   0.0151 FWHM=2.148 RMS=  0.14 (#fit stars= 396 xy_c=1.12 as_c=7.79 #refstars=945)
3255624        coj2m002-ep06-20260708-0986-e92.fits: 2026-07-08T14:21:01 gp 92  24.0856 +/-   0.0147 FWHM=2.486 RMS=  0.13 (#fit stars= 303 xy_c=1.21 as_c=5.15 #refstars=720)
3255628        coj2m002-ep06-20260708-0987-e92.fits: 2026-07-08T14:24:36 gp 92  24.1172 +/-   0.0141 FWHM=2.514 RMS=  0.13 (#fit stars= 318 xy_c=3.50 as_c=12.64 #refstars=732)
3255630        coj2m002-ep06-20260708-0988-e92.fits: 2026-07-08T14:28:11

WARNING [22/Jul/2026 16:48:19] More than 2 observations of Block id=35261 - cannot retrieve all BLKUIDs


35261: 4225822 812644691,812719302
35340: 4227129 813013481
3248041        coj2m002-ep06-20260624-0173-e92.fits: 2026-06-24T13:29:03 gp 92  24.4148 +/-   0.0185 FWHM=2.273 RMS=  0.19 (#fit stars= 290 xy_c=1.78 as_c=4.31 #refstars=608)
3248039        coj2m002-ep06-20260624-0174-e92.fits: 2026-06-24T13:30:09 gp 92  24.4538 +/-   0.0187 FWHM=2.377 RMS=  0.16 (#fit stars= 301 xy_c=1.27 as_c=5.03 #refstars=640)
3248048        coj2m002-ep06-20260624-0175-e92.fits: 2026-06-24T13:31:15 gp 92  24.4718 +/-   0.0179 FWHM=2.092 RMS=  0.16 (#fit stars= 308 xy_c=2.59 as_c=4.57 #refstars=626)
3248052        coj2m002-ep06-20260624-0176-e92.fits: 2026-06-24T13:32:20 gp 92  24.3805 +/-   0.0173 FWHM=2.301 RMS=  0.18 (#fit stars= 286 xy_c=1.69 as_c=6.57 #refstars=600)
3248056        coj2m002-ep06-20260624-0177-e92.fits: 2026-06-24T13:33:25 gp 92  24.4268 +/-   0.0219 FWHM=2.380 RMS=  0.19 (#fit stars= 300 xy_c=3.63 as_c=6.48 #refstars=661)
3248062        coj2m002-ep06-20260624-0178-e92.fits: 2026-06-24T1

In [20]:
##check Block UID ###
obs_block_15 = Block.objects.filter(calibsource= fields[14], num_observed__gte=1) 
block = obs_block_15[0]

block_15_IUDS = []
for blocks in obs_block_15:
    block_15_IUDS.append(blocks.get_blockuid)

a_q_t_15= []

selected_uids = block_15_IUDS   # only the first two UIDs

for f in fields:
    obs_blocks = Block.objects.filter(calibsource=f, num_observed__gte=1)

    for b in obs_blocks:
        if b.get_blockuid not in selected_uids:
            continue

        dataroot = os.path.join(settings.DATA_ROOT, "Hera", b.get_blockdayobs)
        t = summarize_block_quality(dataroot, b)

        if t is not None:
            t["Field Name"] = np.full(len(t), f.name)
            t["# block"] = np.full(len(t), b.id)
            t["Block IUD"] = np.full(len(t), b.get_blockuid)
            a_q_t_15.append(t)

a_q_t_15 = vstack(a_q_t_15)



WARNING [22/Jul/2026 16:49:12] More than 2 observations of Block id=35261 - cannot retrieve all BLKUIDs


35393: 4227153 812982002
3250776        coj2m002-ep07-20260624-0050-e92.fits: 2026-06-24T10:58:32 rp 92  24.6701 +/-   0.0245 FWHM=2.144 RMS=  0.18 (#fit stars= 370 xy_c=-99.00 as_c=-99.00 #refstars=-99)
3250783        coj2m002-ep07-20260624-0051-e92.fits: 2026-06-24T10:59:38 rp 92  24.5950 +/-   0.0267 FWHM=2.327 RMS=  0.18 (#fit stars= 335 xy_c=-99.00 as_c=-99.00 #refstars=-99)
3250779        coj2m002-ep07-20260624-0052-e92.fits: 2026-06-24T11:00:43 rp 92  24.6411 +/-   0.0267 FWHM=2.215 RMS=  0.18 (#fit stars= 353 xy_c=-99.00 as_c=-99.00 #refstars=-99)
3250785        coj2m002-ep07-20260624-0053-e92.fits: 2026-06-24T11:01:48 rp 92  24.6052 +/-   0.0248 FWHM=2.216 RMS=  0.18 (#fit stars= 351 xy_c=-99.00 as_c=-99.00 #refstars=-99)
3250780        coj2m002-ep07-20260624-0054-e92.fits: 2026-06-24T11:02:54 rp 92  24.6058 +/-   0.0241 FWHM=2.296 RMS=  0.18 (#fit stars= 364 xy_c=-99.00 as_c=-99.00 #refstars=-99)
3250782        coj2m002-ep07-20260624-0055-e92.fits: 2026-06-24T11:04:01 rp 92  

In [21]:
##check Block UID ###
obs_block_15 = Block.objects.filter(calibsource= fields[14], num_observed__gte=1) 
block = obs_block_15[0]

block_15_IUDS = []
for blocks in obs_block_15:
    block_15_IUDS.append(blocks.get_blockuid)

a_q_t_15= []

selected_uids = block_15_IUDS[1][0]   # only the first two UIDs


for b in obs_block_15:
    if b.get_blockuid[0] not in selected_uids:
        continue

    dataroot = os.path.join(settings.DATA_ROOT, "Hera", b.get_blockdayobs)
    t = summarize_block_quality(dataroot, b)

    if t is not None:
        t["Field Name"] = np.full(len(t), f.name)
        t["# block"] = np.full(len(t), b.id)
        t["Block IUD"] = np.full(len(t), b.get_blockuid)
        a_q_t_15.append(t)

a_q_t_15 = vstack(a_q_t_15)



35346: 4227135 812964819
3248450        coj2m002-ep06-20260624-0071-e92.fits: 2026-06-24T09:47:22 gp 92  24.2075 +/-   0.0188 FWHM=2.515 RMS=  0.05 (#fit stars= 147 xy_c=3.62 as_c=3.45 #refstars=469)
3248449        coj2m002-ep06-20260624-0072-e92.fits: 2026-06-24T09:48:31 gp 92  24.7800 +/-   0.0337 FWHM=2.668 RMS=-99.00 (#fit stars= 116 xy_c=1.49 as_c=1.95 #refstars=531)
3248454        coj2m002-ep06-20260624-0073-e92.fits: 2026-06-24T09:49:36 gp 92  24.2312 +/-   0.0161 FWHM=2.502 RMS=  0.10 (#fit stars= 150 xy_c=2.32 as_c=3.82 #refstars=530)
3248452        coj2m002-ep06-20260624-0074-e92.fits: 2026-06-24T09:50:42 gp 92  24.7305 +/-   0.0429 FWHM=2.624 RMS=-99.00 (#fit stars= 110 xy_c=1.93 as_c=1.39 #refstars=527)
3248461        coj2m002-ep06-20260624-0075-e92.fits: 2026-06-24T09:51:51 gp 92  24.7962 +/-   0.0458 FWHM=2.990 RMS=-99.00 (#fit stars=  87 xy_c=2.42 as_c=1.18 #refstars=494)
3248464        coj2m002-ep06-20260624-0076-e92.fits: 2026-06-24T09:52:56 gp 92  24.2075 +/-   0.0187

In [22]:
print(master_astro_qual_table)


#organize by smallest to largst rms 
#Low RMS means the observed and predicted positions are close


master_astro_qual_table.sort('Fit RMS')
master_astro_qual_table["Fit RMS"]


#next: get rid of -99's as those indicate no valid read on fit
#Also only keep values with fwhm < 3
# & and indicator 
#mask &= <condition --- True only if mask AND condition are true
# | is the or operator

#intially start as all true 
mask = np.ones(len(master_astro_qual_table), dtype=bool)

for col in master_astro_qual_table.colnames:
    mask &= (master_astro_qual_table[col] != -99.0) #constrcut boolean mask where -99 = FALSE

master_astro_qual_table = master_astro_qual_table[mask] #table with "TRUE" rows remain

m_a_q_table_reduced_1 = master_astro_qual_table[
    (master_astro_qual_table["FWHM"] < 2.8) & 
    (master_astro_qual_table["ZP err"] < 0.28)]


#save as .xlsx
data_ref_images = m_a_q_table_reduced_1.to_pandas()
data_ref_images.to_excel("/home/mwalker/git/neo_assignments/Reference_Image_List_7_10.xlsx", index = False)


           frame filename                     midpoint          ... # block
------------------------------------ -------------------------- ... -------
coj2m002-ep06-20260708-0983-e92.fits 2026-07-08 14:10:16.942000 ...   35772
coj2m002-ep06-20260708-0984-e92.fits 2026-07-08 14:13:51.964000 ...   35772
coj2m002-ep06-20260708-0985-e92.fits 2026-07-08 14:17:26.846000 ...   35772
coj2m002-ep06-20260708-0986-e92.fits 2026-07-08 14:21:01.764000 ...   35772
coj2m002-ep06-20260708-0987-e92.fits 2026-07-08 14:24:36.666000 ...   35772
coj2m002-ep06-20260708-0988-e92.fits 2026-07-08 14:28:11.515000 ...   35772
coj2m002-ep06-20260708-0989-e92.fits 2026-07-08 14:31:50.263000 ...   35772
coj2m002-ep06-20260708-0990-e92.fits 2026-07-08 14:35:25.280000 ...   35772
coj2m002-ep06-20260708-0991-e92.fits 2026-07-08 14:39:00.370000 ...   35772
coj2m002-ep07-20260708-0359-e92.fits 2026-07-08 14:10:16.939000 ...   35772
                                 ...                        ... ...     ...
coj2m002-ep0

/home/mwalker/venv/neocode39_psfphot_venv/lib64/python3.9/site-packages/astropy/table/column.py:345: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = getattr(super(Column, self), op)(other)


In [23]:
#select all row with dates 7/10 and 7/8 with filter ==rp


master_astro_qual_table.sort('Fit RMS')
master_astro_qual_table["Fit RMS"]


#next: get rid of -99's as those indicate no valid read on fit
#Also only keep values with fwhm < 3
# & and indicator 
#mask &= <condition --- True only if mask AND condition are true
# | is the or operator

#intially start as all true 
mask = np.ones(len(master_astro_qual_table), dtype=bool)

for col in master_astro_qual_table.colnames:
    mask &= (master_astro_qual_table[col] != -99.0) #constrcut boolean mask where -99 = FALSE

master_astro_qual_table = master_astro_qual_table[mask] #table with "TRUE" rows remain

m_a_q_table_reduced_1 = master_astro_qual_table[
    (master_astro_qual_table["FWHM"] < 2.8) & 
    (master_astro_qual_table["ZP err"] < 0.28)]

mask = np.array([
    (t.datetime.month == 7) and
    (t.datetime.day in (8, 10)) and
    (f == "rp")
    for t, f in zip(
        m_a_q_table_reduced_1["midpoint"],
        m_a_q_table_reduced_1["filter"]
    )
])

m_a_q_table_reduced_1 = m_a_q_table_reduced_1[mask]
#save as .xlsx
data_ref_images = m_a_q_table_reduced_1.to_pandas()
data_ref_images.to_excel("/home/mwalker/git/neo_assignments/Reference_Image_List_7_8_10.xlsx", index = False)

In [24]:
### Filter out if difference between ref/ fitted stars is too great ###

diff_star_mask = m_a_q_table_reduced_1["num fit stars"] >= (0.5 * (m_a_q_table_reduced_1["num ref stars"]))  #if less than 50 % of stars fitted -- get rid of those rows

m_a_q_table_reduced_2 = m_a_q_table_reduced_1[diff_star_mask]

print(len(m_a_q_table_reduced_2["frame filename"]))

m_a_q_table_reduced_2[m_a_q_table_reduced_2["filter"]=="rp"].to_pandas().to_excel("/home/mwalker/git/neo_assignments/Reference_Image_list_frac_0.5_stars.xlsx")

1


In [25]:
print( " total frames {} \n ---------------\n " 
    " Frames with fhwm <3 and valid RMS and ZP_err <0.25: {}".format(len(master_astro_qual_table), len(m_a_q_table_reduced_1)))
print(m_a_q_table_reduced_1.colnames)

 total frames 1188 
 ---------------
  Frames with fhwm <3 and valid RMS and ZP_err <0.25: 46
['frame filename', 'midpoint', 'filter', 'ZP', 'ZP err', 'FWHM', 'Fit RMS', 'num fit stars', 'XY contrast', 'AS contrast', 'num ref stars', 'Field Name', '# block']


### Extract FWHM and ZP plots ###

In [26]:
### Extract fwhm and zero point plots for all reference fields ###

### from our reduced table: 
###----organze by ref field: parse out string number from that coloumn and then organize table that way 
### Select where filter = ['rp]
### input ["tomato"]
###LAST: dump pngs into the directories: home/mwalker/git/plots/fwhm_plots/filter_rp_tomato and /home/mwalker/git/plots/zero_point+plots/filter_rp_tomato

In [27]:
#file_paths

file_p_fwhm = "/home/mwalker/git/plots/fwhm_plots/filter_rp_tomato"
file_p_zp = "/home/mwalker/git/plots/zero_point+plots/filter_rp_tomato"


#select fields and block numbers in lieu of Static.objects.filter( source_type = StaticSource.REFERENCE_FIELD, name__contains__ ='COJ 2026 Field)
#already filtered out files

coj_fields = m_a_q_table_reduced_1["Field Name"]
block_nums_w_ref = m_a_q_table_reduced_1["# block"]

#We will filter out by field and by block
fields_plot = [StaticSource.objects.get(name = name) for name in coj_fields] #need Query object
blocks_plot = [Block.objects.get(pk =block_id) for block_id in block_nums_w_ref] # gte list of ind blocks

#plot fwhm and zp with remaiing files

generalized_fwhm_plotter(fields_plot,['rp'],['tomato'], file_p_fwhm, False, True)   #indi_block_plots = False, Night_sky_plot = True 
generalized_zeropoint_plotter(fields_plot, ["rp"], ["tomato"], file_p_zp, False, True)


#now do blocks 
block_fp_zp = "/home/mwalker/git/plots/zero_point+plots/filt_rp_tom_ind_block/"
block_fp_fwhm = "/home/mwalker/git/plots/fwhm_plots/filt_rp_tom_ind_blocks/"

generalized_fwhm_plotter(blocks_plot,["rp"],["tomato"],block_fp_fwhm,True,True,)
generalized_zeropoint_plotter(blocks_plot,["rp"],["tomato"],block_fp_zp,True,True,)

/home/mwalker/git/neoexchange/neoexchange/core/plots.py:1312: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  fig, ax = plt.subplots(nrows = 1, tight_layout = True)


['/home/mwalker/git/plots/zero_point+plots/filt_rp_tom_ind_block/single_block_ZP_plot_2026-07-08_Field_#03.png',
 '/home/mwalker/git/plots/zero_point+plots/filt_rp_tom_ind_block/single_block_ZP_plot_2026-07-08_Field_#03.png',
 '/home/mwalker/git/plots/zero_point+plots/filt_rp_tom_ind_block/single_block_ZP_plot_2026-07-08_Field_#02.png',
 '/home/mwalker/git/plots/zero_point+plots/filt_rp_tom_ind_block/single_block_ZP_plot_2026-07-08_Field_#02.png',
 '/home/mwalker/git/plots/zero_point+plots/filt_rp_tom_ind_block/single_block_ZP_plot_2026-07-08_Field_#02.png',
 '/home/mwalker/git/plots/zero_point+plots/filt_rp_tom_ind_block/single_block_ZP_plot_2026-07-08_Field_#02.png',
 '/home/mwalker/git/plots/zero_point+plots/filt_rp_tom_ind_block/single_block_ZP_plot_2026-07-08_Field_#02.png',
 '/home/mwalker/git/plots/zero_point+plots/filt_rp_tom_ind_block/single_block_ZP_plot_2026-07-08_Field_#02.png',
 '/home/mwalker/git/plots/zero_point+plots/filt_rp_tom_ind_block/single_block_ZP_plot_2026-07-08

### get Fits files for ds9 + other analysis ###

In [28]:
fits_files_ds9 = m_a_q_table_reduced_1["frame filename"][m_a_q_table_reduced_1["filter"] == 'rp']
blocks_ds9 = m_a_q_table_reduced_1["# block"][m_a_q_table_reduced_1["filter"] == 'rp' ]

# fields_plot = [
#     StaticSource.objects.get(name=name)
#     for name in fits_files_ds9
# ]
# fp_ds9_fits = "/home/mwalker/git/fits_ds9_images" #destination


print("file names for ds9: {}".format(fits_files_ds9))

file names for ds9:            frame filename           
------------------------------------
coj2m002-ep07-20260708-0357-e92.fits
coj2m002-ep07-20260708-0352-e92.fits
coj2m002-ep07-20260708-0364-e92.fits
coj2m002-ep07-20260708-0366-e92.fits
coj2m002-ep07-20260708-0365-e92.fits
coj2m002-ep07-20260708-0363-e92.fits
coj2m002-ep07-20260708-0362-e92.fits
coj2m002-ep07-20260708-0361-e92.fits
coj2m002-ep07-20260708-0360-e92.fits
coj2m002-ep07-20260710-0047-e92.fits
                                 ...
coj2m002-ep07-20260708-0351-e92.fits
coj2m002-ep07-20260710-0033-e92.fits
coj2m002-ep07-20260710-0028-e92.fits
coj2m002-ep07-20260710-0030-e92.fits
coj2m002-ep07-20260710-0031-e92.fits
coj2m002-ep07-20260710-0036-e92.fits
coj2m002-ep07-20260710-0037-e92.fits
coj2m002-ep07-20260710-0034-e92.fits
coj2m002-ep07-20260710-0035-e92.fits
coj2m002-ep07-20260710-0029-e92.fits
coj2m002-ep07-20260710-0032-e92.fits
Length = 46 rows


In [29]:

dest_dir = "/home/mwalker/git/fits_ds9_images"

for filename, block_id in zip(
    fits_files_ds9,
    blocks_ds9,
):
    block = Block.objects.get(pk=block_id)

    source_dir = os.path.join(
        settings.DATA_ROOT,
        "Hera",
        block.get_blockdayobs
    )

    src = os.path.join(source_dir, filename)
    dst = os.path.join(dest_dir, filename)

    if os.path.exists(src):
        shutil.copy2(src, dst)
    else:
        print(f"Missing: {src}")

In [30]:

###make dirctories for the field blocks ###

unique_field_names = np.unique(m_a_q_table_reduced_1["Field Name"])
unique_block_names = np.unique(m_a_q_table_reduced_1["# block"])
base_dir = "/home/mwalker/git/fits_ds9_images/"

for field in unique_field_names:
    field_num = field.split("#")[-1].strip()   #get ref num and rm whitspace
    mask = (
        (m_a_q_table_reduced_1["filter"] == "rp") &
        (m_a_q_table_reduced_1["Field Name"] == field)
    )

    fits_files_ds9= m_a_q_table_reduced_1["frame filename"][mask]
    blocks_ds9 = m_a_q_table_reduced_1["# block"][mask]



    dir_name = os.path.join(base_dir, f"fits_ds9_images_{field_num}") #create directory for each ref field/ block
    os.makedirs(dir_name, exist_ok= True)

    for filename, block_id in zip(
        fits_files_ds9,
        blocks_ds9,
    ):
        block = Block.objects.get(pk=block_id)

        source_dir = os.path.join(
            settings.DATA_ROOT,
            "Hera",
            block.get_blockdayobs
        )

        src = os.path.join(source_dir, filename)
        dst = os.path.join(dir_name, filename)

        if os.path.exists(src):
            shutil.copy2(src, dst)       #copy of src at dst
        else:
            print(f"Missing: {src}")
    print("file names for ds9: {}".format(fits_files_ds9))



#If nine fits files present == ALL BLOCKS OBSERVED

file names for ds9:            frame filename           
------------------------------------
coj2m002-ep07-20260708-0364-e92.fits
coj2m002-ep07-20260708-0366-e92.fits
coj2m002-ep07-20260708-0365-e92.fits
coj2m002-ep07-20260708-0363-e92.fits
coj2m002-ep07-20260708-0362-e92.fits
coj2m002-ep07-20260708-0361-e92.fits
coj2m002-ep07-20260708-0360-e92.fits
coj2m002-ep07-20260708-0359-e92.fits
file names for ds9:            frame filename           
------------------------------------
coj2m002-ep07-20260708-0357-e92.fits
coj2m002-ep07-20260708-0352-e92.fits
coj2m002-ep07-20260708-0355-e92.fits
coj2m002-ep07-20260708-0358-e92.fits
coj2m002-ep07-20260708-0354-e92.fits
coj2m002-ep07-20260708-0350-e92.fits
coj2m002-ep07-20260708-0353-e92.fits
coj2m002-ep07-20260708-0351-e92.fits
file names for ds9:            frame filename           
------------------------------------
coj2m002-ep07-20260710-0047-e92.fits
coj2m002-ep07-20260710-0039-e92.fits
coj2m002-ep07-20260710-0046-e92.fits
coj2m002-ep07-2

## Build Reference Frames...e-93's ## 

In [31]:
#core/make_reference_fields.py ---bash


## Run DIA on Didymos Science Images ##

In [32]:
#core/make_subractions -- bash

#terminal command: make_subtractions (block_num) (path_with_processed data) (--filter (rp,zp,ip)) (--excecute)

## Photometry ---> Light Curve ##

In [33]:
#core/pipeline_astrometry
#core/pipeline_psf_photometry
#core/light_curve_extraction
#core/lc_analysis